<a href="https://colab.research.google.com/github/440g/painkiller/blob/DataCleaning_re/DataPreprocessing_Re.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

#Train

In [ ]:
train_df = pd.read_csv('../datasets/original/train.csv')

##Data Exploration

In [ ]:
train_df.shape

In [ ]:
for col in train_df.columns:
    print(col)

In [ ]:
train_df.isnull().sum()

##Data Cleaning

###Removing irrelevant observations or duplicates

In [ ]:
#Removing rows with columns that must be non-negative
columns_to_check = [
    'id',
    'n_tokens_title',
    'n_tokens_content',
    'n_unique_tokens',
    'n_non_stop_words',
    'n_non_stop_unique_tokens',
    'num_hrefs',
    'num_self_hrefs',
    'num_imgs',
    'num_videos',
    'average_token_length',
    'num_keywords',
    'self_reference_min_shares',
    'self_reference_max_shares',
    'self_reference_avg_sharess',
    'LDA_00',
    'LDA_01',
    'LDA_02',
    'LDA_03',
    'LDA_04',
    'global_rate_positive_words',
    'global_rate_negative_words',
    'rate_positive_words',
    'rate_negative_words',
    'abs_title_subjectivity',
    'abs_title_sentiment_polarity',
    'shares',
    'y'
]

mask_negative = (train_df[columns_to_check] < 0).any(axis=1)

print(f"Rows with negative values(to remove): {mask_negative.sum()}")

In [ ]:
#Removing irrelevant columns
X_train = train_df.drop(["y", "shares"], axis=1)
y_train = train_df["y"]

In [ ]:
X_train_identifier = X_train['id']
X_train_features = X_train.drop(columns=['id'])

In [ ]:
#Drop duplicated rows based on feature columns
initial_rows = X_train_features.shape[0]
print(f"Initial row count: {initial_rows}")
X_train_features_deduplicated = X_train_features.drop_duplicates()
dropped_duplicates_count = initial_rows - X_train_features_deduplicated.shape[0]
print(f"Duplicated rows count: {dropped_duplicates_count}")
print(f"Removed {dropped_duplicates_count} duplicate rows based on features.")

###Fix structural errors

In [ ]:
#Check data types
print("Data types of features:")
print(X_train_features.dtypes.value_counts())

####Mapping object columns

In [ ]:
#Identify object or categorical columns
object_columns = X_train_features.select_dtypes(include=['object']).columns

print(f"Object columns found: {list(object_columns)}")

for col in object_columns:
  print(f"\nUnique values in '{col}'(excluding NaN): {X_train_features[col].nunique()}")
  print(X_train_features[col].value_counts(dropna=False))

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(object_columns):
    plt.subplot(1, 2, i+1)
    train_df[col].value_counts().plot(kind='bar')
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
#Mapping for data_channel (manually assigned labels 0–5)
data_channel_mapping = {
    'World': 0,
    'Lifestyle': 1,
    'Tech': 2,
    'Entertainment': 3,
    'Business': 4,
    'Social Media': 5
}

#Mapping for weekday (Sunday as 0 through Saturday as 6)
weekday_mapping = {
    'Sunday': 0,
    'Monday': 1,
    'Tuesday': 2,
    'Wednesday': 3,
    'Thursday': 4,
    'Friday': 5,
    'Saturday': 6
}

X_train_features['data_channel'] = X_train_features['data_channel'].map(data_channel_mapping)
X_train_features['weekday'] = X_train_features['weekday'].map(weekday_mapping)

####Checking boolean-like columns

In [ ]:
if X_train_features[col].nunique() <= 3:
  print(f"{col} might be a boolean-like column.")
  print(X_train_features[col].unique())
else:
  print("no boolean-like column.")

###Handling missing values

#####Filling in missing values of 'LDA'

In [ ]:
#Sums of 'LDA_00', 'LDA_01', 'LDA_02', 'LDA_03', 'LDA_04' should be 1
lda_cols = ['LDA_00', 'LDA_01', 'LDA_02', 'LDA_03', 'LDA_04']

#Check rows where LDA sum > 1
lda_sums = X_train_features[lda_cols].sum(axis=1)
mask_over_1 = lda_sums > 1 + 1e-4  # to avoid floating point issues

lda_over_rows = X_train_features[mask_over_1]

print(f"Number of rows where LDA sum > 1: {mask_over_1.sum()}")

In [ ]:
# Fill single NaN in LDA columns
def fill_lda(row):
    nulls = row[lda_cols].isnull()

    if nulls.sum() == 1:
        known_sum = row[lda_cols][~nulls].sum()
        row[lda_cols] = row[lda_cols].fillna(1.0 - known_sum)
    return row

X_train_features = X_train_features.apply(fill_lda, axis=1)

#Check remaining rows
invalid_rows = X_train_features[lda_cols].sum(axis=1).round(5) != 1
print(f"Number of rows where LDA sum is not 1: {invalid_rows.sum()}")

In [ ]:
#Count NaN in remaining rows
lda_nan_counts = X_train_features[lda_cols].isnull().sum(axis=1)

lda_nan_2 = lda_nan_counts == 2
lda_nan_3 = lda_nan_counts == 3
lda_nan_4 = lda_nan_counts == 4
lda_nan_5 = lda_nan_counts == 5

rows_with_2_nan = X_train_features[lda_nan_2]
rows_with_3_nan = X_train_features[lda_nan_3]
rows_with_4_nan = X_train_features[lda_nan_4]
rows_with_5_nan = X_train_features[lda_nan_5]

print(f"Rows with 2 NaNs: {rows_with_2_nan.shape[0]}")
print(f"Rows with 3 NaNs: {rows_with_3_nan.shape[0]}")
print(f"Rows with 4 NaNs: {rows_with_4_nan.shape[0]}")
print(f"Rows with 5 NaNs: {rows_with_5_nan.shape[0]}")

In [ ]:
#Delete rows with more than 4 NaNs since there is a high degree of uncertainty

original_row_count = X_train_features.shape[0]
X_train_features = X_train_features[~lda_nan_4]

print(f"Remaining rows after deletion: {X_train_features.shape[0]} (from {original_row_count})")

y_train = y_train.loc[~lda_nan_4]

print(f"Remaining rows in y_train: {y_train.shape[0]} (should match {X_train_features.shape[0]})")

In [ ]:
#Fill 2~3 NaNs in LDA columns by equal distribution
mask_2_or_3_nan = (lda_nan_counts == 2) | (lda_nan_counts == 3)
rows_to_fill = X_train_features.loc[mask_2_or_3_nan]

def fill_equal_lda(row):
    nulls = row[lda_cols].isnull()
    n_nulls = nulls.sum()

    if n_nulls in [2, 3]:
        known_sum = row[lda_cols][~nulls].sum()
        remaining = 1.0 - known_sum
        fill_value = remaining / n_nulls
        row[lda_cols] = row[lda_cols].fillna(fill_value)

    return row

rows_filled = rows_to_fill.apply(fill_equal_lda, axis=1)

X_train_features.loc[mask_2_or_3_nan] = rows_filled

#Check if all rows now sum to ~1
lda_sums = X_train_features[lda_cols].sum(axis=1).round(5)
invalid_lda_rows = X_train_features[lda_sums != 1]
print(f"Rows where LDA sum is still not 1: {len(invalid_lda_rows)}")

#####Filling in missing values using feature correlation

In [ ]:
#Finding high correlation features
corr_matrix = X_train_features.corr().abs()
high_corr_pairs = []
seen = set()

for i, j in combinations(corr_matrix.columns, 2):
    corr_val = corr_matrix.loc[i, j]
    if corr_val >= 0.8:
        pair = tuple(sorted([i, j]))
        if pair not in seen:
            seen.add(pair)
            high_corr_pairs.append((pair[0], pair[1], corr_val))
            print(f"{pair[0]} ↔ {pair[1]} (corr: {corr_val:.3f})")

In [ ]:
for feat_a, feat_b, corr_val in high_corr_pairs:
    # feat_a has missing values, feat_b is complete
    mask_train = X_train_features[feat_a].notna() & X_train_features[feat_b].notna()
    mask_pred  = X_train_features[feat_a].isna()  & X_train_features[feat_b].notna()

    if mask_pred.sum() > 0:
        lr = LinearRegression()
        lr.fit(X_train_features.loc[mask_train, [feat_b]],
               X_train_features.loc[mask_train, feat_a])

        X_train_features.loc[mask_pred, feat_a] = lr.predict(
            X_train_features.loc[mask_pred, [feat_b]]
        )
        print(f"Imputed missing '{feat_a}' using '{feat_b}'  ({mask_pred.sum()} rows)")

    # feat_b has missing values, feat_a is complete
    mask_train = X_train_features[feat_b].notna() & X_train_features[feat_a].notna()
    mask_pred  = X_train_features[feat_b].isna()  & X_train_features[feat_a].notna()

    if mask_pred.sum() > 0:
        lr = LinearRegression()
        lr.fit(X_train_features.loc[mask_train, [feat_a]],
               X_train_features.loc[mask_train, feat_b])
        X_train_features.loc[mask_pred, feat_b] = lr.predict(
            X_train_features.loc[mask_pred, [feat_a]]
        )
        print(f"Imputed missing '{feat_b}' using '{feat_a}'  ({mask_pred.sum()} rows)")

In [ ]:
# Checking remaining missing values in imputed features
imputed_features = set()
for feat_a, feat_b, _ in high_corr_pairs:
    imputed_features.add(feat_a)
    imputed_features.add(feat_b)

remaining_missing = X_train_features[list(imputed_features)].isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if remaining_missing.empty:
    print("no missing values remaining in imputed features")
else:
    print("imputed features with missing values:")
    for col, cnt in remaining_missing.items():
        print(f"   - {col}: {cnt}")

####Filling in missing values using variance (consider dropping)

In [ ]:
variances = X_train_features.var()

sorted_variances = variances.sort_values(ascending=False)

print("Feature variances (highest to lowest):")
for feature, var in sorted_variances.items():
    print(f"  - {feature}: {var:.6f}")

In [ ]:
#Impute missing values with the column median
low_var_thresh = 0.01
low_var_feats = variances[variances <= low_var_thresh].index
print("Low-variance features:")
print(list(low_var_feats))
print("\n")

for col in low_var_feats:
    med = X_train_features[col].median()
    n_missing = X_train_features[col].isna().sum()
    if n_missing > 0:
        X_train_features[col] = X_train_features[col].fillna(med)
        print(f"Filled {n_missing} missing values in '{col}' with median {med}")

####Filling in missing values of object columns if they have low correlation with y_train

In [ ]:
# Pearson correlation between 'weekday' and target y_train
weekday_corr = X_train_features['weekday'].corr(y_train)
print(f"Correlation between 'weekday' and y_train: {weekday_corr:.4f}")

# Pearson correlation between 'data_channel' (original count) and y_train
data_channel_corr  = X_train_features['data_channel'].corr(y_train)
print(f"Correlation between 'data_channel' and y_train: {data_channel_corr:.4f}")

In [ ]:
weekday_before = X_train_features['weekday'].copy()

In [ ]:
# Filling missing values randomly while maintaining the distribution
np.random.seed(42)
# Impute only the 'weekday' feature since 'data_channel' shows non-negligible correlation with target y_train(0.1983)
col = 'weekday'
mask = X_train_features[col].isna()
observed = X_train_features.loc[~mask, col].values
fills = np.random.choice(observed, size=mask.sum(), replace=True)
X_train_features.loc[mask, col] = fills

In [ ]:
weekday_after = X_train_features['weekday']

In [ ]:
#Compare before/after filling in the missing values
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Before
axs[0].hist(weekday_before.dropna(), bins=range(8), align='left', edgecolor='black')
axs[0].set_title("Weekday Before Imputation")
axs[0].set_xlabel("Weekday (0=Sun … 6=Sat)")
axs[0].set_ylabel("Count")
axs[0].set_xticks(range(7))

# After
axs[1].hist(weekday_after, bins=range(8), align='left', edgecolor='black')
axs[1].set_title("Weekday After Imputation")
axs[1].set_xlabel("Weekday (0=Sun … 6=Sat)")
axs[1].set_xticks(range(7))

plt.tight_layout()
plt.show()

####Filling in missing values of skewed features

In [ ]:
# For each missing entry, fill 0 with probability of 0, otherwise fill the median of nonzero.
features_to_keep_raw = [
    'n_unique_tokens', 'n_non_stop_unique_tokens',
    'average_token_length', 'kw_min_avg', 'global_subjectivity',
    'rate_positive_words', 'rate_negative_words',
    'avg_positive_polarity', 'avg_negative_polarity',
    'title_subjectivity', 'abs_title_sentiment_polarity'
]

In [ ]:
np.random.seed(42)

for col in features_to_keep_raw:
    data = X_train_features[col].dropna()
    hist, edges = np.histogram(data, bins=50)
    centers = (edges[:-1] + edges[1:]) / 2
    idx = np.argmin(np.abs(centers - 0))
    p_bin = hist[idx] / len(data)
    bin_mask = (data >= edges[idx]) & (data < edges[idx+1])
    bin_vals = data[bin_mask]
    bin_mode = pd.Series(bin_vals).mode()[0] if len(bin_vals) else pd.Series(data).mode()[0]
    other_median = pd.Series(data[~bin_mask]).median() if len(data[~bin_mask]) else bin_mode

    na_mask = X_train_features[col].isna()
    n_missing = na_mask.sum()
    if n_missing:
        fills = np.random.choice(
            [bin_mode, other_median],
            size=n_missing,
            p=[p_bin, 1-p_bin]
        )
        X_train_features.loc[na_mask, col] = fills

    print(f"Imputed '{col}'")

####Filling in missing values with mean

In [ ]:
# Data spread evenly
mean_cols = [
    'max_positive_polarity',
    'min_negative_polarity',
]

In [ ]:
for col in mean_cols:
    mean_val = X_train_features[col].mean()
    X_train_features[col] = X_train_features[col].fillna(mean_val)
    print(f"Filled missing values in '{col}'")

In [ ]:
# int
mean_cols = [
    'data_channel'
]

In [ ]:
for col in mean_cols:
    mean_val = X_train_features[col].mean()
    mean_int = int(round(mean_val))
    X_train_features[col] = X_train_features[col].fillna(mean_int)
    print(f"Filled missing values in '{col}' with rounded mean: {mean_int}")

####Filling in missing values with median

In [ ]:
# Skewed normal distribution form
median_cols = [
    'n_tokens_content',
    'num_hrefs',
    'num_self_hrefs',
    'num_imgs',
    'kw_avg_max',
    'kw_avg_avg'
]

In [ ]:
for col in median_cols:
    med_val = X_train_features[col].median()
    X_train_features[col] = X_train_features[col].fillna(med_val)
    print(f"Filled missing values in '{col}'")

####Filling in missing values with mode

In [ ]:
#There are values that appear noticeably more
mode_cols = [
    'num_videos',
    'kw_min_min',
    'kw_max_min',
    'kw_avg_min',
    'kw_min_max',
    'kw_max_max',
    'kw_max_avg',
    'self_reference_min_shares',
    'self_reference_max_shares',
    'self_reference_avg_sharess',
    'title_sentiment_polarity',
    'abs_title_subjectivity'
]

In [ ]:
for col in mode_cols:
    mode_ser = X_train_features[col].mode(dropna=True)
    if not mode_ser.empty:
        mode_val = mode_ser[0]
        X_train_features[col] = X_train_features[col].fillna(mode_val)
        print(f"Filled missing values in '{col}'")
    else:
        print(f"No mode found for '{col}', skipping.")

####Others

* Filling in missing values of 'n_tokens_title' column

In [ ]:
shares_corr  = train_df['n_tokens_title'].corr(y_train)
print(f"Correlation between 'n_tokens_title' and y_train: {shares_corr:.4f}")

In [ ]:
# Filling missing values randomly while maintaining the distribution
np.random.seed(42)

freq = X_train_features['n_tokens_title'].value_counts(normalize=True)
# Select only those values whose frequency is ≥ 0.1% (excluding possible outliers)
common_vals = freq[freq >= 0.001].index
mask = X_train_features['n_tokens_title'].isna()
observed_common = X_train_features.loc[~mask & X_train_features['n_tokens_title'].isin(common_vals),
                                'n_tokens_title'].values
fills = np.random.choice(observed_common, size=mask.sum(), replace=True)
X_train_features.loc[mask, 'n_tokens_title'] = fills

* Filling in missing values of 'num_keywords' column

In [ ]:
shares_corr  = train_df['num_keywords'].corr(y_train)
print(f"Correlation between 'num_keywords' and y_train: {shares_corr:.4f}")

In [ ]:
# Filling missing values randomly while maintaining the distribution
np.random.seed(42)

freq = X_train_features['num_keywords'].value_counts(normalize=True)
# Select only those values whose frequency is ≥ 0.1% (excluding possible outliers)
common_vals = freq[freq >= 0.001].index
mask = X_train_features['num_keywords'].isna()
observed_common = X_train_features.loc[~mask & X_train_features['num_keywords'].isin(common_vals),
                                'num_keywords'].values
fills = np.random.choice(observed_common, size=mask.sum(), replace=True)
X_train_features.loc[mask, 'num_keywords'] = fills

* Filling in missing values of 'n_non_stop_words' column

In [ ]:
# Fill missing 'n_non_stop_words' values with 0.999999995
# since most of the values range in 0.99999999 to 0.999999999
X_train_features['n_non_stop_words'] = X_train_features['n_non_stop_words'].fillna(0.999999995)

n_missing_after = X_train_features['n_non_stop_words'].isna().sum()

###Handling Outliers

In [ ]:
numeric_columns = X_train_features.select_dtypes(include=np.number).columns
outlier_summary = {}
print(f"Numerical columns for outlier detection:")
for i, col in enumerate(numeric_columns, 1):
    print(f"{i}. {col}")

####IQR Clip

#####Tukey IQR (k=1.5)

In [ ]:
#Detecting outliers using Tukey IQR (k=1.5)
def detect_outliers_tukey(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers, lower, upper

outlier_summary_tukey = {}
print("Outlier analysis (Tukey(k=1.5)):")

for col in numeric_columns:
    outliers, lower, upper = detect_outliers_tukey(X_train_features, col)
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(X_train_features)) * 100

    outlier_summary_tukey[col] = {
        'count': outlier_count,
        'percentage': outlier_percentage,
        'lower': lower,
        'upper': upper
    }

    print(f"{col}: {outlier_count} outliers. ({outlier_percentage:.2f}%)")

# Columns with high percentage outliers
high_outlier_cols = [col for col, info in outlier_summary_tukey.items()
                     if info['percentage'] > 10]
print(f"\nColumns with more than 10% outliers:")
for i, col in enumerate(high_outlier_cols, 1):
    print(f"{col}")

#####Carling IQR

In [ ]:
#Detecting outliers using Carling's adaptive IQR method
def detect_outliers_carling(df, column):
    n = len(df[column])
    Q2 = df[column].median()
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    # Carling's adaptive multiplier formula
    k = (17.63 * n - 23.64) / (7.74 * n - 3.71)
    lower = Q2 - k * IQR
    upper = Q2 + k * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers, lower, upper

outlier_summary_carling = {}
print("Outlier analysis (Carling):")

for col in numeric_columns:
    outliers, lower, upper = detect_outliers_carling(X_train_features, col)
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(X_train_features)) * 100

    outlier_summary_carling[col] = {
        'count': outlier_count,
        'percentage': outlier_percentage,
        'lower': lower,
        'upper': upper
    }

    print(f"{col}: {outlier_count} outliers. ({outlier_percentage:.2f}%)")

# Columns with high percentage outliers
high_outlier_cols = [col for col, info in outlier_summary_carling.items()
                    if info['percentage'] > 10]
print(f"\nColumns with more than 10% outliers:")
for i, col in enumerate(high_outlier_cols, 1):
    print(f"{col}")

#####Boxplot

In [ ]:
sns.set(style="whitegrid")

cols_per_row = 2
total_cols = len(numeric_columns)
rows = math.ceil(total_cols / cols_per_row)
plt.figure(figsize=(10, 44))

for i, col in enumerate(numeric_columns):
    plt.subplot(rows, cols_per_row, i + 1)
    sns.boxplot(x=X_train_features[col], color='lightgrey')

    tukey_lower = outlier_summary_tukey[col]['lower']
    tukey_upper = outlier_summary_tukey[col]['upper']
    plt.axvline(tukey_lower, color='blue', linestyle='--', label='Tukey lower', alpha=0.7)
    plt.axvline(tukey_upper, color='blue', linestyle='--', label='Tukey upper', alpha=0.7)

    carling_lower = outlier_summary_carling[col]['lower']
    carling_upper = outlier_summary_carling[col]['upper']
    plt.axvline(carling_lower, color='red', linestyle=':', label='Carling lower', alpha=0.7)
    plt.axvline(carling_upper, color='red', linestyle=':', label='Carling upper', alpha=0.7)

    plt.title(col)
    if i == 0:
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

#####Clip

In [ ]:
# Tukey‐clip (k=1.5)
tukey_cols = [
    'n_tokens_title', 'global_subjectivity',
    'min_positive_polarity', 'max_positive_polarity',
    'min_negative_polarity', 'max_negative_polarity',
    'title_subjectivity', 'title_sentiment_polarity',
    'abs_title_subjectivity', 'abs_title_sentiment_polarity'
]

for col in tukey_cols:
    clipped = X_train_features[col].clip(lower=0)
    transformed = np.log1p(clipped)
    Q1 = X_train_features[col].quantile(0.25)
    Q3 = X_train_features[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    X_train_features[col] = X_train_features[col].clip(lower, upper)

# log1p‐transform, Tukey‐clip
log1p_tukey_cols = [
    'n_tokens_content', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos',
    'rate_positive_words', 'rate_negative_words', 'global_rate_positive_words',
    'global_rate_negative_words', 'kw_min_min', 'kw_max_min', 'kw_avg_min', 'kw_min_max',
    'kw_max_max', 'kw_avg_max', 'kw_min_avg', 'kw_max_avg', 'kw_avg_avg',
    'self_reference_min_shares', 'self_reference_max_shares', 'self_reference_avg_sharess'
]

for col in log1p_tukey_cols:
    clipped = X_train_features[col].clip(lower=0)
    log_transformed = np.log1p(clipped)
    Q1 = log_transformed.quantile(0.25)
    Q3 = log_transformed.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    X_train_features[col] = log_transformed.clip(lower, upper)

# Carling‐clip
col = 'num_keywords'
Q1 = X_train_features[col].quantile(0.25)
Q3 = X_train_features[col].quantile(0.75)
IQR = Q3 - Q1
n = len(X_train_features[col])
k = (17.63 * n - 23.64) / (7.74 * n - 3.71)
lower, upper = Q1 - k*IQR, Q3 + k*IQR
X_train_features[col] = X_train_features[col].clip(lower, upper)

# Clip to [0,1]
clip01_cols = [f'LDA_{i:02d}' for i in range(5)] + [
    'n_unique_tokens', 'n_non_stop_unique_tokens'
]
for col in clip01_cols:
    X_train_features[col] = X_train_features[col].clip(0.0, 1.0)

# Clip to [-1,1]
clipm11_cols = [
    'global_sentiment_polarity',
    'avg_positive_polarity', 'avg_negative_polarity'
]
for col in clipm11_cols:
    X_train_features[col] = X_train_features[col].clip(-1.0, 1.0)

##Data Split

In [ ]:
X_train = X_train_features.copy()
X_train.insert(0, 'id', X_train_identifier)

X_train.to_csv('../datasets/X_train_preprocessed.csv', index=False)
y_train.to_csv('../datasets/y_train_preprocessed.csv', index=False)

In [ ]:
X_train_model = X_train.drop(columns=['id'])
y_train_model = y_train

X_train_re, X_train_val_re, y_train_re, y_train_val_re = train_test_split(
    X_train_model,
    y_train_model,
    test_size=0.2,
    random_state=42,
    stratify=y_train_model
)

X_train_re.to_csv('../datasets/preprocessed/X_train.csv', index=False)
y_train_re.to_csv('../datasets/preprocessed/y_train.csv', index=False)

X_train_val_re.to_csv('../datasets/preprocessed/X_val.csv', index=False)
y_train_val_re.to_csv('../datasets/preprocessed/y_val.csv', index=False)

# Test

In [ ]:
test_df = pd.read_csv('../datasets/original/test.csv')

##Data Exploration

In [ ]:
test_df.shape

In [ ]:
for col in test_df.columns:
    print(col)

In [ ]:
test_df.isnull().sum()

##Data Cleaning

###Removing irrelevant observations or duplicates

In [ ]:
#Removing rows with columns that must be non-negative
columns_to_check = [
    'id',
    'n_tokens_title',
    'n_tokens_content',
    'n_unique_tokens',
    'n_non_stop_words',
    'n_non_stop_unique_tokens',
    'num_hrefs',
    'num_self_hrefs',
    'num_imgs',
    'num_videos',
    'average_token_length',
    'num_keywords',
    'self_reference_min_shares',
    'self_reference_max_shares',
    'self_reference_avg_sharess',
    'LDA_00',
    'LDA_01',
    'LDA_02',
    'LDA_03',
    'LDA_04',
    'global_rate_positive_words',
    'global_rate_negative_words',
    'rate_positive_words',
    'rate_negative_words',
    'abs_title_subjectivity',
    'abs_title_sentiment_polarity'
]

mask_negative = (test_df[columns_to_check] < 0).any(axis=1)

print(f"Rows with negative values(to remove): {mask_negative.sum()}")

In [ ]:
X_test = test_df
X_test_identifier = X_test['id']
X_test_features = X_test.drop(columns=['id'])

In [ ]:
# Drop duplicates based on feature columns
initial_rows = X_test_features.shape[0]
print(f"Initial row count: {initial_rows}")
X_test_features_deduplicated = X_test_features.drop_duplicates()
dropped_duplicates_count = initial_rows - X_test_features_deduplicated.shape[0]
print(f"Duplicated rows count: {dropped_duplicates_count}")
print(f"Removed {dropped_duplicates_count} duplicate rows based on features.")

###Fix structural errors

In [ ]:
# Check data types
print("Data types of features:")
print(X_test_features.dtypes.value_counts())

####Mapping object columns

In [ ]:
# Identify object or categorical columns
object_columns = X_test_features.select_dtypes(include=['object']).columns

print(f"Object columns found: {list(object_columns)}")
for col in object_columns:
  print(f"\nUnique values in '{col}': {X_test_features[col].nunique()}")
  print(X_test_features[col].value_counts(dropna=False))

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(object_columns):
    plt.subplot(1, 2, i+1)
    test_df[col].value_counts().plot(kind='bar')
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Mapping for data_channel (manually assigned labels 0–5)
data_channel_mapping = {
    'World': 0,
    'Lifestyle': 1,
    'Tech': 2,
    'Entertainment': 3,
    'Business': 4,
    'Social Media': 5
}

# Mapping for weekday (Sunday as 0 through Saturday as 6)
weekday_mapping = {
    'Sunday': 0,
    'Monday': 1,
    'Tuesday': 2,
    'Wednesday': 3,
    'Thursday': 4,
    'Friday': 5,
    'Saturday': 6
}

X_test_features['data_channel'] = X_test_features['data_channel'].map(data_channel_mapping)
X_test_features['weekday'] = X_test_features['weekday'].map(weekday_mapping)

####Checking boolean-like columns

In [ ]:
if X_test_features[col].nunique() <= 3:
  print(f"{col} might be a boolean-like column.")
  print(X_test_features[col].unique())
else:
  print("no boolean-like column.")

###Handling missing values

#####Filling in missing values of 'LDA'

In [ ]:
#Sums of 'LDA_00', 'LDA_01', 'LDA_02', 'LDA_03', 'LDA_04' should be 1
lda_cols = ['LDA_00', 'LDA_01', 'LDA_02', 'LDA_03', 'LDA_04']

#Check rows where LDA sum > 1
lda_sums = X_test_features[lda_cols].sum(axis=1)
mask_over_1 = lda_sums > 1 + 1e-4  # to avoid floating point issues

lda_over_rows = X_test_features[mask_over_1]

print(f"Number of rows where LDA sum > 1: {mask_over_1.sum()}")

In [ ]:
# Fill single NaN in LDA columns
def fill_lda(row):
    nulls = row[lda_cols].isnull()

    if nulls.sum() == 1:
        known_sum = row[lda_cols][~nulls].sum()
        row[lda_cols] = row[lda_cols].fillna(1.0 - known_sum)
    return row

X_test_features = X_test_features.apply(fill_lda, axis=1)

#Check remaining rows
invalid_rows = X_test_features[lda_cols].sum(axis=1).round(5) != 1
print(f"Number of rows where LDA sum is not 1: {invalid_rows.sum()}")

In [ ]:
#Count NaN in remaining rows
lda_nan_counts = X_test_features[lda_cols].isnull().sum(axis=1)

lda_nan_2 = lda_nan_counts == 2
lda_nan_3 = lda_nan_counts == 3
lda_nan_4 = lda_nan_counts == 4
lda_nan_5 = lda_nan_counts == 5

rows_with_2_nan = X_test_features[lda_nan_2]
rows_with_3_nan = X_test_features[lda_nan_3]
rows_with_4_nan = X_test_features[lda_nan_4]
rows_with_5_nan = X_test_features[lda_nan_5]

print(f"Rows with 2 NaNs: {rows_with_2_nan.shape[0]}")
print(f"Rows with 3 NaNs: {rows_with_3_nan.shape[0]}")
print(f"Rows with 4 NaNs: {rows_with_4_nan.shape[0]}")
print(f"Rows with 5 NaNs: {rows_with_5_nan.shape[0]}")

In [ ]:
#Fill NaNs in LDA columns by equal distribution(should't exclude 4, 5 NaNs)
def fill_lda(row):
    current_sum = row[lda_cols].sum(skipna=True)
    num_missing = row[lda_cols].isna().sum()

    if num_missing > 0:
        remaining = 1 - current_sum
        fill_value = remaining / num_missing
        row[lda_cols] = row[lda_cols].fillna(fill_value)

    return row

X_test_features = X_test_features.apply(fill_lda, axis=1)

#####Filling in missing values using feature correlation

In [ ]:
#Finding high correlation features
corr_matrix = X_test_features.corr().abs()
high_corr_pairs = []
seen = set()

for i, j in combinations(corr_matrix.columns, 2):
    corr_val = corr_matrix.loc[i, j]
    if corr_val >= 0.8:
        pair = tuple(sorted([i, j]))
        if pair not in seen:
            seen.add(pair)
            high_corr_pairs.append((pair[0], pair[1], corr_val))
            print(f"{pair[0]} ↔ {pair[1]} (corr: {corr_val:.3f})")

In [ ]:
for feat_a, feat_b, corr_val in high_corr_pairs:
    # feat_a has missing values, feat_b is complete
    mask_train = X_test_features[feat_a].notna() & X_test_features[feat_b].notna()
    mask_pred  = X_test_features[feat_a].isna()  & X_test_features[feat_b].notna()

    if mask_pred.sum() > 0:
        lr = LinearRegression()
        lr.fit(X_test_features.loc[mask_train, [feat_b]],
               X_test_features.loc[mask_train, feat_a])

        X_test_features.loc[mask_pred, feat_a] = lr.predict(
            X_test_features.loc[mask_pred, [feat_b]]
        )
        print(f"Imputed missing '{feat_a}' using '{feat_b}'  ({mask_pred.sum()} rows)")

    # feat_b has missing values, feat_a is complete
    mask_train = X_test_features[feat_b].notna() & X_test_features[feat_a].notna()
    mask_pred  = X_test_features[feat_b].isna()  & X_test_features[feat_a].notna()

    if mask_pred.sum() > 0:
        lr = LinearRegression()
        lr.fit(X_test_features.loc[mask_train, [feat_a]],
               X_test_features.loc[mask_train, feat_b])
        X_test_features.loc[mask_pred, feat_b] = lr.predict(
            X_test_features.loc[mask_pred, [feat_a]]
        )
        print(f"Imputed missing '{feat_b}' using '{feat_a}'  ({mask_pred.sum()} rows)")

In [ ]:
# Checking remaining missing values in imputed features
imputed_features = set()
for feat_a, feat_b, _ in high_corr_pairs:
    imputed_features.add(feat_a)
    imputed_features.add(feat_b)

remaining_missing = X_test_features[list(imputed_features)].isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if remaining_missing.empty:
    print("no missing values remaining in imputed features")
else:
    print("imputed features with missing values:")
    for col, cnt in remaining_missing.items():
        print(f"   - {col}: {cnt}")

####Filling in missing values using variance (consider dropping)

In [ ]:
variances = X_test_features.var()

sorted_variances = variances.sort_values(ascending=False)

print("Feature variances (highest to lowest):")
for feature, var in sorted_variances.items():
    print(f"  - {feature}: {var:.6f}")

In [ ]:
#Impute missing values with the column median
low_var_thresh = 0.01
low_var_feats = variances[variances <= low_var_thresh].index
print("Low-variance features:")
print(list(low_var_feats))
print("\n")

for col in low_var_feats:
    med = X_test_features[col].median()
    n_missing = X_test_features[col].isna().sum()
    if n_missing > 0:
        X_test_features[col] = X_test_features[col].fillna(med)
        print(f"Filled {n_missing} missing values in '{col}' with median {med}")

####Filling in missing values of object columns if they have low correlation with y_test

In [ ]:
weekday_before = X_test_features['weekday'].copy()

In [ ]:
# Filling missing values randomly while maintaining the distribution
np.random.seed(42)
# Impute only the 'weekday' feature since 'data_channel' shows non-negligible correlation with target y_test(0.1983)
col = 'weekday'
mask = X_test_features[col].isna()
observed = X_test_features.loc[~mask, col].values
fills = np.random.choice(observed, size=mask.sum(), replace=True)
X_test_features.loc[mask, col] = fills

In [ ]:
weekday_after = X_test_features['weekday']

In [ ]:
#Compare before/after filling in the missing values
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Before
axs[0].hist(weekday_before.dropna(), bins=range(8), align='left', edgecolor='black')
axs[0].set_title("Weekday Before Imputation")
axs[0].set_xlabel("Weekday (0=Sun … 6=Sat)")
axs[0].set_ylabel("Count")
axs[0].set_xticks(range(7))

# After
axs[1].hist(weekday_after, bins=range(8), align='left', edgecolor='black')
axs[1].set_title("Weekday After Imputation")
axs[1].set_xlabel("Weekday (0=Sun … 6=Sat)")
axs[1].set_xticks(range(7))

plt.tight_layout()
plt.show()

####Filling in missing values of skewed features

In [ ]:
# For each missing entry, fill 0 with probability of 0, otherwise fill the median of nonzero.
features_to_keep_raw = [
    'n_unique_tokens', 'n_non_stop_unique_tokens',
    'average_token_length', 'kw_min_avg', 'global_subjectivity',
    'rate_positive_words', 'rate_negative_words',
    'avg_positive_polarity', 'avg_negative_polarity',
    'title_subjectivity', 'abs_title_sentiment_polarity'
]

In [ ]:
np.random.seed(42)

for col in features_to_keep_raw:
    data = X_test_features[col].dropna()
    hist, edges = np.histogram(data, bins=50)
    centers = (edges[:-1] + edges[1:]) / 2
    idx = np.argmin(np.abs(centers - 0))
    p_bin = hist[idx] / len(data)
    bin_mask = (data >= edges[idx]) & (data < edges[idx+1])
    bin_vals = data[bin_mask]
    bin_mode = pd.Series(bin_vals).mode()[0] if len(bin_vals) else pd.Series(data).mode()[0]
    other_median = pd.Series(data[~bin_mask]).median() if len(data[~bin_mask]) else bin_mode

    na_mask = X_test_features[col].isna()
    n_missing = na_mask.sum()
    if n_missing:
        fills = np.random.choice(
            [bin_mode, other_median],
            size=n_missing,
            p=[p_bin, 1-p_bin]
        )
        X_test_features.loc[na_mask, col] = fills

    print(f"Imputed '{col}'")

####Filling in missing values with mean

In [ ]:
# Data spread evenly
mean_cols = [
    'max_positive_polarity',
    'min_negative_polarity',
]

In [ ]:
for col in mean_cols:
    mean_val = X_test_features[col].mean()
    X_test_features[col] = X_test_features[col].fillna(mean_val)
    print(f"Filled missing values in '{col}'")

In [ ]:
# int
mean_cols = [
    'data_channel'
]

In [ ]:
for col in mean_cols:
    mean_val = X_test_features[col].mean()
    mean_int = int(round(mean_val))
    X_test_features[col] = X_test_features[col].fillna(mean_int)
    print(f"Filled missing values in '{col}' with rounded mean: {mean_int}")

####Filling in missing values with median

In [ ]:
# Skewed normal distribution form
median_cols = [
    'n_tokens_content',
    'num_hrefs',
    'num_self_hrefs',
    'num_imgs',
    'kw_avg_max',
    'kw_avg_avg'
]

In [ ]:
for col in median_cols:
    med_val = X_test_features[col].median()
    X_test_features[col] = X_test_features[col].fillna(med_val)
    print(f"Filled missing values in '{col}'")

####Filling in missing values with mode

In [ ]:
#There are values that appear noticeably more
mode_cols = [
    'num_videos',
    'kw_min_min',
    'kw_max_min',
    'kw_avg_min',
    'kw_min_max',
    'kw_max_max',
    'kw_max_avg',
    'self_reference_min_shares',
    'self_reference_max_shares',
    'self_reference_avg_sharess',
    'title_sentiment_polarity',
    'abs_title_subjectivity'
]

In [ ]:
for col in mode_cols:
    mode_ser = X_test_features[col].mode(dropna=True)
    if not mode_ser.empty:
        mode_val = mode_ser[0]
        X_test_features[col] = X_test_features[col].fillna(mode_val)
        print(f"Filled missing values in '{col}'")
    else:
        print(f"No mode found for '{col}', skipping.")

####Others

* Filling in missing values of 'n_tokens_title' column

In [ ]:
# Filling missing values randomly while maintaining the distribution
np.random.seed(42)

freq = X_test_features['n_tokens_title'].value_counts(normalize=True)
# Select only those values whose frequency is ≥ 0.1% (excluding possible outliers)
common_vals = freq[freq >= 0.001].index
mask = X_test_features['n_tokens_title'].isna()
observed_common = X_test_features.loc[~mask & X_test_features['n_tokens_title'].isin(common_vals),
                                'n_tokens_title'].values
fills = np.random.choice(observed_common, size=mask.sum(), replace=True)
X_test_features.loc[mask, 'n_tokens_title'] = fills

* Filling in missing values of 'num_keywords' column

In [ ]:
# Filling missing values randomly while maintaining the distribution
np.random.seed(42)

freq = X_test_features['num_keywords'].value_counts(normalize=True)
# Select only those values whose frequency is ≥ 0.1% (excluding possible outliers)
common_vals = freq[freq >= 0.001].index
mask = X_test_features['num_keywords'].isna()
observed_common = X_test_features.loc[~mask & X_test_features['num_keywords'].isin(common_vals),
                                'num_keywords'].values
fills = np.random.choice(observed_common, size=mask.sum(), replace=True)
X_test_features.loc[mask, 'num_keywords'] = fills

* Filling in missing values of 'n_non_stop_words' column

In [ ]:
# Fill missing 'n_non_stop_words' values with 0.999999995
# since most of the values range in 0.99999999 to 0.999999999
X_test_features['n_non_stop_words'] = X_test_features['n_non_stop_words'].fillna(0.999999995)

n_missing_after = X_test_features['n_non_stop_words'].isna().sum()

In [ ]:
X_test_features.isnull().sum()

###Handling Outliers

In [ ]:
numeric_columns = X_test_features.select_dtypes(include=np.number).columns
outlier_summary = {}
print(f"Numerical columns for outlier detection:")
for i, col in enumerate(numeric_columns, 1):
    print(f"{i}. {col}")

####IQR Clip

#####Tukey IQR (k=1.5)

In [ ]:
#Detecting outliers using Tukey IQR (k=1.5)
def detect_outliers_tukey(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers, lower, upper

outlier_summary_tukey = {}
print("Outlier analysis (Tukey(k=1.5)):")

for col in numeric_columns:
    outliers, lower, upper = detect_outliers_tukey(X_test_features, col)
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(X_test_features)) * 100

    outlier_summary_tukey[col] = {
        'count': outlier_count,
        'percentage': outlier_percentage,
        'lower': lower,
        'upper': upper
    }

    print(f"{col}: {outlier_count} outliers. ({outlier_percentage:.2f}%)")

# Columns with high percentage outliers
high_outlier_cols = [col for col, info in outlier_summary_tukey.items()
                     if info['percentage'] > 10]
print(f"\nColumns with more than 10% outliers:")
for i, col in enumerate(high_outlier_cols, 1):
    print(f"{col}")

#####Carling IQR

In [ ]:
#Detecting outliers using Carling's adaptive IQR method
def detect_outliers_carling(df, column):
    n = len(df[column])
    Q2 = df[column].median()
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    # Carling's adaptive multiplier formula
    k = (17.63 * n - 23.64) / (7.74 * n - 3.71)
    lower = Q2 - k * IQR
    upper = Q2 + k * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers, lower, upper

outlier_summary_carling = {}
print("Outlier analysis (Carling):")

for col in numeric_columns:
    outliers, lower, upper = detect_outliers_carling(X_test_features, col)
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(X_test_features)) * 100

    outlier_summary_carling[col] = {
        'count': outlier_count,
        'percentage': outlier_percentage,
        'lower': lower,
        'upper': upper
    }

    print(f"{col}: {outlier_count} outliers. ({outlier_percentage:.2f}%)")

# Columns with high percentage outliers
high_outlier_cols = [col for col, info in outlier_summary_carling.items()
                    if info['percentage'] > 10]
print(f"\nColumns with more than 10% outliers:")
for i, col in enumerate(high_outlier_cols, 1):
    print(f"{col}")

#####Boxplot

In [ ]:
sns.set(style="whitegrid")

cols_per_row = 2
total_cols = len(numeric_columns)
rows = math.ceil(total_cols / cols_per_row)
plt.figure(figsize=(10, 44))

for i, col in enumerate(numeric_columns):
    plt.subplot(rows, cols_per_row, i + 1)
    sns.boxplot(x=X_test_features[col], color='lightgrey')

    tukey_lower = outlier_summary_tukey[col]['lower']
    tukey_upper = outlier_summary_tukey[col]['upper']
    plt.axvline(tukey_lower, color='blue', linestyle='--', label='Tukey lower', alpha=0.7)
    plt.axvline(tukey_upper, color='blue', linestyle='--', label='Tukey upper', alpha=0.7)

    carling_lower = outlier_summary_carling[col]['lower']
    carling_upper = outlier_summary_carling[col]['upper']
    plt.axvline(carling_lower, color='red', linestyle=':', label='Carling lower', alpha=0.7)
    plt.axvline(carling_upper, color='red', linestyle=':', label='Carling upper', alpha=0.7)

    plt.title(col)
    if i == 0:
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

#####Clip

In [ ]:
# Tukey‐clip (k=1.5)
tukey_cols = [
    'n_tokens_title', 'global_subjectivity',
    'min_positive_polarity', 'max_positive_polarity',
    'min_negative_polarity', 'max_negative_polarity',
    'title_subjectivity', 'title_sentiment_polarity',
    'abs_title_subjectivity', 'abs_title_sentiment_polarity'
]

for col in tukey_cols:
    clipped = X_test_features[col].clip(lower=0)
    transformed = np.log1p(clipped)
    Q1 = X_test_features[col].quantile(0.25)
    Q3 = X_test_features[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    X_test_features[col] = X_test_features[col].clip(lower, upper)

# log1p‐transform, Tukey‐clip
log1p_tukey_cols = [
    'n_tokens_content', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos',
    'rate_positive_words', 'rate_negative_words', 'global_rate_positive_words',
    'global_rate_negative_words', 'kw_min_min', 'kw_max_min', 'kw_avg_min', 'kw_min_max',
    'kw_max_max', 'kw_avg_max', 'kw_min_avg', 'kw_max_avg', 'kw_avg_avg',
    'self_reference_min_shares', 'self_reference_max_shares', 'self_reference_avg_sharess'
]

for col in log1p_tukey_cols:
    clipped = X_test_features[col].clip(lower=0)
    log_transformed = np.log1p(clipped)
    Q1 = log_transformed.quantile(0.25)
    Q3 = log_transformed.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    X_test_features[col] = log_transformed.clip(lower, upper)

# Carling‐clip
col = 'num_keywords'
Q1 = X_test_features[col].quantile(0.25)
Q3 = X_test_features[col].quantile(0.75)
IQR = Q3 - Q1
n = len(X_test_features[col])
k = (17.63 * n - 23.64) / (7.74 * n - 3.71)
lower, upper = Q1 - k*IQR, Q3 + k*IQR
X_test_features[col] = X_test_features[col].clip(lower, upper)

# Clip to [0,1]
clip01_cols = [f'LDA_{i:02d}' for i in range(5)] + [
    'n_unique_tokens', 'n_non_stop_unique_tokens'
]
for col in clip01_cols:
    X_test_features[col] = X_test_features[col].clip(0.0, 1.0)

# Clip to [-1,1]
clipm11_cols = [
    'global_sentiment_polarity',
    'avg_positive_polarity', 'avg_negative_polarity'
]
for col in clipm11_cols:
    X_test_features[col] = X_test_features[col].clip(-1.0, 1.0)

##Data Split

In [ ]:
X_test = X_test_features.copy()
X_test.insert(0, 'id', X_test_identifier)

X_test.to_csv('../datasets/preprocessed/X_test.csv', index=False)